# 모듈 05: 메모리와 대화 관리 (Memory)

## 이 모듈에서 배울 것

- 메모리 없는 체인의 한계: 매번 invoke마다 이전 대화를 잊는 문제
- `InMemoryChatMessageHistory`: 대화 기록을 저장하는 저장소
- `MessagesPlaceholder`: 프롬프트에 대화 기록 삽입 위치 지정
- `RunnableWithMessageHistory`: 체인에 메모리를 붙이는 래퍼
- 세션 ID로 여러 대화를 독립적으로 관리하는 방법

## 학습 목표

1. **메모리의 필요성 이해**: 메모리 없는 체인이 왜 대화 맥락을 유지하지 못하는지 직접 확인한다
2. **InMemoryChatMessageHistory + RunnableWithMessageHistory 패턴 습득**: 두 클래스를 조합해서 대화 기록을 유지하는 체인을 만들 수 있다
3. **세션 ID로 다중 대화 관리**: 여러 사용자/대화방을 session_id로 독립적으로 분리하여 관리할 수 있다

## 전체 흐름도

### 메모리 없는 체인 (기존)
```
[invoke 1]  사용자: "내 이름은 민준이야"  --> LLM --> "안녕하세요, 민준님!"
[invoke 2]  사용자: "내 이름이 뭐야?"    --> LLM --> "죄송해요, 모르겠어요"  (기억 없음!)
```

### 메모리 있는 체인 (이 모듈)
```
[invoke 1]  사용자: "내 이름은 민준이야"
                           |  --> InMemoryChatMessageHistory 에 저장
                           v
            LLM --> "안녕하세요, 민준님!"

[invoke 2]  사용자: "내 이름이 뭐야?"
                           |  --> InMemoryChatMessageHistory 에서 이전 대화 로드
                           v
            [system] + [이전 대화 기록] + [현재 질문]  --> LLM --> "민준님이라고 하셨어요!"
```

핵심: **RunnableWithMessageHistory**가 체인과 저장소 사이에서 대화 기록을 자동으로 로드/저장한다.

## 메모리란?

**금붕어 비유**

메모리 없는 LLM은 **3초 기억 금붕어**다.

- invoke()를 호출할 때마다 완전히 새로 시작한다
- 직전에 무슨 말을 했는지 전혀 모른다
- 아무리 자연스럽게 대화해도, 내부적으로는 매번 처음 만나는 사람이다

```
사용자: "내 이름은 민준이야"   --> AI: "안녕하세요, 민준님!"
          (기억 초기화)
사용자: "내 이름이 뭐야?"      --> AI: "저는 당신의 이름을 모릅니다"  <-- 금붕어!
```

해결책: 이전 대화 내용을 **메시지 목록**으로 모아서 매번 프롬프트에 함께 넘겨주면 된다.
`InMemoryChatMessageHistory`가 그 메시지 목록 저장소이고,
`RunnableWithMessageHistory`가 그 저장소에서 꺼내 프롬프트에 주입하는 역할을 한다.

In [12]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# .env 파일 경로: 05_memory/ 폴더 기준 상위 디렉토리
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.dirname(notebook_dir)
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

api_key = os.getenv('GOOGLE_API_KEY')
if api_key and api_key != 'your_api_key_here':
    print(f'[OK] GOOGLE_API_KEY 로드 성공: {api_key[:10]}...')
else:
    print('[FAIL] API 키가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요.')

# temperature=0.7: 대화형 AI는 약간 높은 온도로 자연스러운 대화 유도
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7)
print('[OK] ChatGoogleGenerativeAI 초기화 완료 (temperature=0.7)')

[OK] GOOGLE_API_KEY 로드 성공: AIzaSyAlrJ...
[OK] ChatGoogleGenerativeAI 초기화 완료 (temperature=0.7)


## 메모리 없는 체인의 문제

먼저 기존 방식(메모리 없는 체인)으로 연속 대화를 시도해보자.

첫 번째 invoke에서 이름을 알려준 뒤,
두 번째 invoke에서 이름을 물어보면 어떻게 될까?

각 invoke는 **독립적**이므로 이전 대화가 전혀 전달되지 않는다.
LLM 입장에서는 두 번째 질문이 아무 맥락 없이 날아온 것이다.

In [13]:
# 메모리 없는 기본 체인
prompt_no_memory = ChatPromptTemplate.from_messages([
    ('system', '당신은 친절한 어시스턴트입니다.'),
    ('human', '{input}'),
])
chain_no_memory = prompt_no_memory | llm | StrOutputParser()

r1 = chain_no_memory.invoke({'input': '제 이름은 민준이에요.'})
print(f'AI (1번째): {r1}')

r2 = chain_no_memory.invoke({'input': '제 이름이 뭔지 기억하세요?'})
print(f'AI (2번째): {r2}')

print('\n[확인] 메모리 없는 체인은 이전 대화를 기억하지 못합니다.')

AI (1번째): 안녕하세요, 민준님! 만나서 반갑습니다. 😊 무엇을 도와드릴까요?
AI (2번째): 아니요, 죄송합니다만 저는 인공지능이라서 개인적인 정보를 기억하거나 저장하지 않습니다.

이전 대화에서 말씀해주셨더라도 제가 따로 저장해 두지 않기 때문에 기억하지 못합니다.

혹시 이번 대화에서 저에게 이름을 알려주시면, 지금부터는 그 이름으로 불러드릴 수 있습니다! 😊

[확인] 메모리 없는 체인은 이전 대화를 기억하지 못합니다.


## InMemoryChatMessageHistory

**편지 묶음 비유**

`InMemoryChatMessageHistory`는 대화 기록을 담는 **편지 묶음**이다.

- 사람이 보낸 편지(`HumanMessage`)와 AI가 보낸 답장(`AIMessage`)을 순서대로 쌓아둔다
- 새 대화를 시작할 때 이 편지 묶음 전체를 꺼내서 LLM에게 "이전에 나눈 얘기야"라고 건네준다
- LLM은 편지 묶음을 읽고 맥락을 파악한 뒤 다음 답장을 쓴다

```
history.messages = [
    HumanMessage(content='안녕, 나는 민준이야'),
    AIMessage(content='안녕하세요, 민준님!'),
    HumanMessage(content='내 이름이 뭐야?'),
    AIMessage(content='민준님이라고 하셨어요!'),
]
```

주의: `InMemoryChatMessageHistory`는 메모리(RAM)에만 저장되므로 프로그램을 재시작하면 사라진다.
영구 저장이 필요하면 Redis, 데이터베이스 기반 히스토리 클래스를 사용한다.

In [14]:
# InMemoryChatMessageHistory 직접 조작
history = InMemoryChatMessageHistory()

history.add_user_message('안녕하세요, 제 이름은 민준입니다.')
history.add_ai_message('안녕하세요, 민준님! 반갑습니다.')
history.add_user_message('저는 파이썬을 공부하고 있어요.')
history.add_ai_message('파이썬은 훌륭한 선택입니다!')

print(f'저장된 메시지 수: {len(history.messages)}개')
for msg in history.messages:
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    print(f'  [{role}] {msg.content}')

print('[OK] InMemoryChatMessageHistory: 대화 기록 추가 및 조회 성공')

저장된 메시지 수: 4개
  [Human] 안녕하세요, 제 이름은 민준입니다.
  [AI] 안녕하세요, 민준님! 반갑습니다.
  [Human] 저는 파이썬을 공부하고 있어요.
  [AI] 파이썬은 훌륭한 선택입니다!
[OK] InMemoryChatMessageHistory: 대화 기록 추가 및 조회 성공


In [15]:
# MessagesPlaceholder: 프롬프트에 대화 기록이 들어갈 "빈 칸" 지정
prompt_with_history = ChatPromptTemplate.from_messages([
    ('system', '당신은 친절한 한국어 어시스턴트입니다. 이전 대화를 기억합니다.'),
    MessagesPlaceholder(variable_name='chat_history'),  # 대화 기록이 여기 삽입됨
    ('human', '{input}'),
])

# 프롬프트 구조 확인
print('프롬프트 메시지 구조:')
for msg in prompt_with_history.messages:
    print(f'  {type(msg).__name__}: {msg}')

print('[OK] MessagesPlaceholder로 chat_history 삽입 위치 지정 완료')

프롬프트 메시지 구조:
  SystemMessagePromptTemplate: prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 친절한 한국어 어시스턴트입니다. 이전 대화를 기억합니다.') additional_kwargs={}
  MessagesPlaceholder: variable_name='chat_history'
  HumanMessagePromptTemplate: prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}') additional_kwargs={}
[OK] MessagesPlaceholder로 chat_history 삽입 위치 지정 완료


## RunnableWithMessageHistory

`RunnableWithMessageHistory`는 **체인과 메모리 저장소를 연결하는 래퍼**다.

**작동 방식**:
1. `invoke()` 호출 시 `config`에서 `session_id`를 읽는다
2. `get_session_history(session_id)`를 호출해서 해당 세션의 대화 기록을 가져온다
3. 대화 기록을 `chat_history` 자리에 주입해서 프롬프트를 완성한다
4. LLM을 실행하고 응답을 받는다
5. **자동으로** 이번 사용자 입력과 AI 응답을 저장소에 추가한다

```
chain_with_history = RunnableWithMessageHistory(
    chain,                        # 기존 체인
    get_session_history,          # 세션 ID -> 저장소 반환 함수
    input_messages_key='input',   # 입력 dict에서 사용자 메시지 키
    history_messages_key='chat_history',  # 프롬프트에서 대화 기록 키 (MessagesPlaceholder와 일치)
)
```

개발자가 직접 `history.add_user_message()`를 호출할 필요 없다. 래퍼가 자동으로 처리한다.

In [16]:
# 세션 저장소: session_id -> InMemoryChatMessageHistory
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """세션 ID에 해당하는 대화 기록 저장소를 반환. 없으면 새로 생성."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

print('store dict 초기 상태:', store)
print('get_session_history("test") 호출 후:', get_session_history('test'))
print('store dict 상태:', list(store.keys()))
print('[OK] store + get_session_history 함수 정의 완료')

store dict 초기 상태: {}
get_session_history("test") 호출 후: 
store dict 상태: ['test']
[OK] store + get_session_history 함수 정의 완료


In [17]:
# chain + RunnableWithMessageHistory 구성
chain = prompt_with_history | llm | StrOutputParser()

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history',
)

# 첫 번째 대화: 이름 알려주기
config = {'configurable': {'session_id': 'user_001'}}
r1 = chain_with_history.invoke({'input': '제 이름은 민준이에요.'}, config=config)
print(f'AI (1번째): {r1}')

AI (1번째): 안녕하세요, 민준님! 만나서 반갑습니다. 무엇을 도와드릴까요?


In [18]:
# 두 번째 대화: 이름 기억 확인 (같은 session_id 사용)
r2 = chain_with_history.invoke({'input': '제 이름이 뭔지 기억하세요?'}, config=config)
print(f'AI (2번째): {r2}')

if '민준' in r2:
    print('[OK] RunnableWithMessageHistory: 이전 대화 기억 성공')
else:
    print('[FAIL] 이름을 기억하지 못했습니다.')

AI (2번째): 네, 민준님! 당연히 기억합니다. 민준님이시죠? 😊
[OK] RunnableWithMessageHistory: 이전 대화 기억 성공


In [19]:
# store에 저장된 메시지 수 + 대화 기록 내용 출력 확인
session_history = get_session_history('user_001')
print(f"store['user_001'] 메시지 수: {len(session_history.messages)}개")
print()
print('--- 저장된 대화 기록 ---')
for i, msg in enumerate(session_history.messages, 1):
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    print(f'  {i}. [{role}] {msg.content[:80]}')

# 메시지 수 검증: 1번째 invoke 2개 (human+ai) + 2번째 invoke 2개 = 4개
expected_count = 4
actual_count = len(session_history.messages)
if actual_count == expected_count:
    print(f'[OK] 메시지 수 확인: {actual_count}개 (예상 {expected_count}개)')
else:
    print(f'[확인] 메시지 수: {actual_count}개 (예상 {expected_count}개)')

store['user_001'] 메시지 수: 4개

--- 저장된 대화 기록 ---
  1. [Human] 제 이름은 민준이에요.
  2. [AI] 안녕하세요, 민준님! 만나서 반갑습니다. 무엇을 도와드릴까요?
  3. [Human] 제 이름이 뭔지 기억하세요?
  4. [AI] 네, 민준님! 당연히 기억합니다. 민준님이시죠? 😊
[OK] 메시지 수 확인: 4개 (예상 4개)


## 세션 ID로 다중 대화 관리

**대화방 번호 비유**

`session_id`는 **대화방 번호**다.

- `user_001` 대화방에서 민준이가 대화한다
- `user_002` 대화방에서 지수가 대화한다
- 두 대화방은 완전히 독립적이다. 한 방의 대화가 다른 방으로 섞이지 않는다

```
store = {
    'user_001': InMemoryChatMessageHistory([...민준이 대화...]),
    'user_002': InMemoryChatMessageHistory([...지수 대화...]),
}
```

실제 서비스에서 session_id는 보통 사용자 ID, 채팅방 ID, UUID 등을 사용한다.
동일한 체인 객체를 재사용하면서 config의 session_id만 바꿔주면 된다.

In [20]:
# 새로운 store로 세션 독립성 확인 (이전 실습과 분리)
store_multi = {}

def get_session_history_multi(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store_multi:
        store_multi[session_id] = InMemoryChatMessageHistory()
    return store_multi[session_id]

chain_multi = RunnableWithMessageHistory(
    prompt_with_history | llm | StrOutputParser(),
    get_session_history_multi,
    input_messages_key='input',
    history_messages_key='chat_history',
)

config_001 = {'configurable': {'session_id': 'user_001'}}
config_002 = {'configurable': {'session_id': 'user_002'}}

# user_001 세션: 민준
chain_multi.invoke({'input': '안녕, 나는 민준이야.'}, config=config_001)
print('[user_001] 이름 등록: 민준')

# user_002 세션: 지수
chain_multi.invoke({'input': '안녕, 나는 지수야.'}, config=config_002)
print('[user_002] 이름 등록: 지수')

# 각 세션에서 이름 확인
r_001 = chain_multi.invoke({'input': '내 이름이 뭐야?'}, config=config_001)
r_002 = chain_multi.invoke({'input': '내 이름이 뭐야?'}, config=config_002)

print(f'\nuser_001 응답: {r_001}')
print(f'user_002 응답: {r_002}')

[user_001] 이름 등록: 민준
[user_002] 이름 등록: 지수

user_001 응답: 민준님이십니다.
user_002 응답: 지수님이세요! 😊


In [21]:
# 두 세션 대화 기록 비교: 각 세션이 서로 독립적임을 확인
h_001 = get_session_history_multi('user_001')
h_002 = get_session_history_multi('user_002')

print(f'user_001 메시지 수: {len(h_001.messages)}개')
print(f'user_002 메시지 수: {len(h_002.messages)}개')

print('\n--- user_001 대화 기록 ---')
for msg in h_001.messages:
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    print(f'  [{role}] {msg.content[:60]}')

print('\n--- user_002 대화 기록 ---')
for msg in h_002.messages:
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    print(f'  [{role}] {msg.content[:60]}')

# 세션 독립성 검증
if '민준' in r_001 and '지수' in r_002:
    print('\n[OK] 세션 독립성 확인: 두 세션이 서로 독립적으로 관리됨')
else:
    print('\n[FAIL] 세션이 혼용되었습니다.')

user_001 메시지 수: 4개
user_002 메시지 수: 4개

--- user_001 대화 기록 ---
  [Human] 안녕, 나는 민준이야.
  [AI] 안녕하세요, 민준님! 만나서 반갑습니다. 무엇을 도와드릴까요?
  [Human] 내 이름이 뭐야?
  [AI] 민준님이십니다.

--- user_002 대화 기록 ---
  [Human] 안녕, 나는 지수야.
  [AI] 안녕하세요, 지수님! 만나서 반갑습니다. 무엇을 도와드릴까요? 😊
  [Human] 내 이름이 뭐야?
  [AI] 지수님이세요! 😊

[OK] 세션 독립성 확인: 두 세션이 서로 독립적으로 관리됨


## 배운 것 정리

### 핵심 개념

**메모리**: LLM 체인이 이전 대화 내용을 기억하도록 대화 기록을 저장하고 프롬프트에 주입하는 패턴

### 세 가지 핵심 클래스

| 클래스 | 역할 |
|--------|------|
| `InMemoryChatMessageHistory` | 대화 기록 저장소 (편지 묶음) |
| `MessagesPlaceholder` | 프롬프트에 대화 기록 삽입 위치 지정 |
| `RunnableWithMessageHistory` | 체인과 저장소를 연결하는 래퍼 |

### 핵심 코드 패턴

**1단계: 세션 저장소 준비**
```python
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]
```

**2단계: 프롬프트에 MessagesPlaceholder 추가**
```python
prompt = ChatPromptTemplate.from_messages([
    ('system', '...'),
    MessagesPlaceholder(variable_name='chat_history'),  # 대화 기록 삽입 위치
    ('human', '{input}'),
])
```

**3단계: RunnableWithMessageHistory로 래핑**
```python
chain = prompt | llm | StrOutputParser()

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history',
)
```

**4단계: config로 세션 지정해서 invoke**
```python
config = {'configurable': {'session_id': 'user_001'}}
result = chain_with_history.invoke({'input': '...'}, config=config)
```

### 기억할 한 줄

> **LLM은 항상 기억을 잃는다. 메모리 패턴은 이전 대화를 프롬프트에 담아 LLM에게 "복기"시키는 방법이다.**

## 다음 모듈 예고: 모듈 06 - RAG (검색 증강 생성)

지금까지 LLM은 학습된 내용만 알고 있었다. 최신 정보나 외부 문서에 대해서는 모른다.

```
사용자: "우리 회사 취업 규칙 3조가 뭐야?"
AI: "죄송합니다, 저는 그 정보를 갖고 있지 않습니다."  <- RAG 없음!
```

모듈 06에서는 **RAG (Retrieval Augmented Generation)**를 배운다:
외부 문서를 벡터 DB에 저장하고, 질문과 관련된 부분을 검색해서 LLM에게 넘겨주는 패턴이다.

### 준비 체크리스트

- [ ] 모듈 05 모든 셀 오류 없이 실행 완료
- [ ] 셀 6: 메모리 없는 체인이 이름을 기억 못함 확인 (문제 시연)
- [ ] 셀 8: `len(history.messages) == 4` 확인
- [ ] 셀 12~13: 연속 대화에서 "민준" 이름을 AI가 기억하여 응답에 포함
- [ ] 셀 14: `store['user_001']` 메시지 수 4개 확인
- [ ] 셀 16~17: user_001은 민준, user_002는 지수 각각 기억, 세션 독립성 확인

---
*모듈 05 완료. 다음: 모듈 06 - RAG*